###Mount Drive & Install All Packages

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("="*60)
print("INSTALLING ALL REQUIRED PACKAGES")
print("="*60)

# Install all packages at once
!pip install -q transformers torch torchvision pillow accelerate
!pip install -q ftfy regex tqdm
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q chromadb
!pip install -q sentence-transformers
!pip install -q streamlit plotly pandas openpyxl
!npm install -g localtunnel

print("\n✅ All packages installed successfully!")
print("="*60)

Mounted at /content/drive
INSTALLING ALL REQUIRED PACKAGES
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 5.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 86.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 82.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 117.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/6

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


###Extract Images from ZIP

In [ ]:
import zipfile
import os
from pathlib import Path
from tqdm import tqdm

print("="*60)
print("STEP 2: EXTRACTING IMAGES")
print("="*60)

# Define paths
zip_path = '/content/drive/MyDrive/images/Dataset_Small.zip'
extract_path = '/content/fashion_images/'

# Check if zip exists
if not os.path.exists(zip_path):
    print(f"❌ ZIP file not found at: {zip_path}")
    print("Please update the path to your ZIP file")
else:
    print(f"✅ Found ZIP file: {zip_path}")

    # Extract if not already done
    if not os.path.exists(extract_path) or len(list(Path(extract_path).rglob('*.*'))) == 0:
        print("📦 Extracting images...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_path)
        print(f"✅ Extracted to: {extract_path}")
    else:
        print(f"✅ Images already extracted at: {extract_path}")

    # Scan for images
    print("\n🔍 Scanning for images...")
    image_paths = []

    extensions = ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']
    for ext in extensions:
        image_paths.extend(list(Path(extract_path).rglob(ext)))

    print(f"✅ Found {len(image_paths)} images")

    if len(image_paths) > 0:
        print(f"\n📸 Sample images:")
        for i, path in enumerate(image_paths[:5]):
            print(f"  {i+1}. {path.name}")
    else:
        print("❌ No images found! Please check your ZIP file contents.")

print("="*60)

STEP 2: EXTRACTING IMAGES
✅ Found ZIP file: /content/drive/MyDrive/images/Dataset_Small.zip
📦 Extracting images...
✅ Extracted to: /content/fashion_images/

🔍 Scanning for images...
✅ Found 10097 images

📸 Sample images:
  1. img_00000002.jpg
  2. img_00000004.jpg
  3. img_00000003.jpg
  4. img_00000001.jpg
  5. img_00000005.jpg


###Generate Captions with BLIP

In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
import torch
from tqdm import tqdm
import pandas as pd
import os
from concurrent.futures import ThreadPoolExecutor

print("="*60)
print("STEP 3: FAST CAPTION GENERATION WITH BLIP")
print("="*60)

# Check prerequisites
if 'image_paths' not in globals() or len(image_paths) == 0:
    print("❌ No images found! Please run Step 2 first.")
else:
    captions_file = '/content/fashion_captions.csv'

    if os.path.exists(captions_file):
        print("📂 Loading existing captions...")
        captions_df = pd.read_csv(captions_file)
        print(f"✅ Loaded {len(captions_df)} existing captions")
    else:
        print(f"🚀 Generating captions for {len(image_paths)} images...")
        print("⏱️ Estimated time: ~5-15 minutes depending on dataset size\n")

        # Load BLIP model (faster than BLIP-2)
        device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"📥 Loading BLIP model on {device}...")

        processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
        model = BlipForConditionalGeneration.from_pretrained(
            "Salesforce/blip-image-captioning-base"
        ).to(device)
        model.eval()

        print("✅ Model loaded\n")

        def load_and_preprocess_image(img_path):
            """Load and prepare image"""
            try:
                img = Image.open(img_path).convert('RGB')
                # Resize for faster processing
                img.thumbnail((384, 384), Image.Resampling.LANCZOS)
                return img
            except Exception as e:
                return None

        def generate_caption_batch(image_paths_batch):
            """Generate captions for a batch of images"""
            results = []

            # Load images in parallel
            with ThreadPoolExecutor(max_workers=4) as executor:
                images = list(executor.map(load_and_preprocess_image, image_paths_batch))

            # Filter valid images
            valid_data = [(img, path) for img, path in zip(images, image_paths_batch) if img is not None]

            if not valid_data:
                return results

            valid_images, valid_paths = zip(*valid_data)

            try:
                # Process batch
                inputs = processor(
                    images=list(valid_images),
                    return_tensors="pt",
                    padding=True
                ).to(device)

                with torch.no_grad():
                    outputs = model.generate(
                        **inputs,
                        max_length=50,
                        num_beams=3,
                        early_stopping=True
                    )

                # Decode captions
                captions = processor.batch_decode(outputs, skip_special_tokens=True)

                # Create results
                for path, caption in zip(valid_paths, captions):
                    results.append({
                        'image_path': str(path),
                        'filename': path.name,
                        'caption': f"Fashion item: {caption}"
                    })

            except Exception as e:
                print(f"\n⚠️ Batch error: {e}")
                # Fallback to individual processing
                for path in valid_paths:
                    results.append({
                        'image_path': str(path),
                        'filename': path.name,
                        'caption': f"Fashion item from {path.stem}"
                    })

            return results

        # Process in batches
        batch_size = 16 if device == "cuda" else 8
        captions_data = []

        total_batches = (len(image_paths) + batch_size - 1) // batch_size
        print(f"📊 Processing in {total_batches} batches of {batch_size}\n")

        for i in tqdm(range(0, len(image_paths), batch_size), desc="Generating captions"):
            batch_paths = image_paths[i:i+batch_size]
            batch_results = generate_caption_batch(batch_paths)
            captions_data.extend(batch_results)

            # Save checkpoint every 500 items
            if len(captions_data) % 500 == 0 and len(captions_data) > 0:
                temp_df = pd.DataFrame(captions_data)
                temp_df.to_csv(captions_file + '.tmp', index=False)
                tqdm.write(f"💾 Checkpoint: {len(captions_data)} captions saved")

        # Create final DataFrame
        captions_df = pd.DataFrame(captions_data)
        captions_df.to_csv(captions_file, index=False)

        # Cleanup
        if os.path.exists(captions_file + '.tmp'):
            os.remove(captions_file + '.tmp')

        del model
        del processor
        torch.cuda.empty_cache()

        print(f"\n✅ Caption generation complete!")
        print(f"   Total captions: {len(captions_df)}")
        print(f"   Success rate: {len(captions_df)/len(image_paths)*100:.1f}%")
        print(f"   Saved to: {captions_file}")

    # Display samples
    print("\n" + "="*60)
    print("📋 SAMPLE CAPTIONS")
    print("="*60)
    for idx, row in captions_df.head(10).iterrows():
        print(f"\n{idx+1}. {row['filename']}")
        print(f"   {row['caption']}")

    print(f"\n📊 Total: {len(captions_df)} captions")
    print("="*60)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


STEP 3: FAST CAPTION GENERATION WITH BLIP
🚀 Generating captions for 10097 images...
⏱️ Estimated time: ~5-15 minutes depending on dataset size

📥 Loading BLIP model on cuda...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

...

💾 Checkpoint: 8000 captions saved


Generating captions:  99%|█████████▉| 625/632 [19:48<00:12,  1.84s/it]

💾 Checkpoint: 10000 captions saved


Generating captions: 100%|██████████| 632/632 [19:59<00:00,  1.90s/it]


✅ Caption generation complete!
   Total captions: 10097
   Success rate: 100.0%
   Saved to: /content/fashion_captions.csv

📋 SAMPLE CAPTIONS

1. img_00000002.jpg
   Fashion item: a woman in leather pants and a white blouse

2. img_00000004.jpg
   Fashion item: a man wearing a black jacket and black pants

3. img_00000003.jpg
   Fashion item: a woman wearing a black top and black pants

4. img_00000001.jpg
   Fashion item: a woman wearing a tank top with the chicago bulls logo on it

5. img_00000005.jpg
   Fashion item: a man wearing a baseball shirt with the bulls logo on it

6. img_00000002.jpg
   Fashion item: a young man wearing a black leather jacket and jeans

7. img_00000004.jpg
   Fashion item: a woman wearing a jacket and jeans

8. img_00000003.jpg
   Fashion item: a women ' s jacket with a fur collar

9. img_00000001.jpg
   Fashion item: a brown jacket with a zipper on the front and a zipper on the back

10. img_00000005.jpg
   Fashion item: a woman walking down the street



###Generate CLIP Embeddings

In [ ]:
import torch
import clip
from PIL import Image
import numpy as np
from tqdm import tqdm
import json
import os
from concurrent.futures import ThreadPoolExecutor
import gc

print("="*60)
print("STEP 4: ULTRA-FAST CLIP EMBEDDINGS")
print("="*60)

# Check prerequisites
if 'captions_df' not in globals():
    if os.path.exists('/content/fashion_captions.csv'):
        captions_df = pd.read_csv('/content/fashion_captions.csv')
        print(f"✅ Loaded {len(captions_df)} captions from file")
    else:
        raise Exception("❌ No captions found! Run Step 3 first.")
else:
    print(f"✅ Found {len(captions_df)} captions in memory")

embeddings_file = '/content/fashion_embeddings.json'

# Check if already exists
if os.path.exists(embeddings_file):
    print(f"\n📂 Loading existing embeddings...")
    with open(embeddings_file, 'r') as f:
        embeddings_data = json.load(f)
    print(f"✅ Loaded {len(embeddings_data)} existing embeddings")
else:
    print("\n🚀 Generating CLIP embeddings...")

    # Load CLIP
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"📥 Loading CLIP model on {device}...")

    clip_model, preprocess = clip.load("ViT-B/32", device=device)
    clip_model.eval()
    print("✅ CLIP loaded\n")

    # Optimize batch size
    batch_size = 128 if device == "cuda" else 32
    print(f"⚙️ Batch size: {batch_size}")

    def load_image_fast(img_path):
        """Fast image loading with error handling"""
        try:
            img = Image.open(img_path).convert('RGB')
            img = img.resize((224, 224), Image.Resampling.BICUBIC)
            return preprocess(img)
        except:
            return None

    @torch.no_grad()
    def generate_embeddings_batch(image_tensors, captions):
        """Generate embeddings for a batch"""
        if len(image_tensors) == 0:
            return np.array([])

        # Image embeddings
        images = torch.stack(image_tensors).to(device)
        image_features = clip_model.encode_image(images)
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)

        # Text embeddings
        text_tokens = clip.tokenize(captions, truncate=True).to(device)
        text_features = clip_model.encode_text(text_tokens)
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)

        # Combine: 70% image + 30% text
        combined = 0.7 * image_features + 0.3 * text_features
        combined = combined / combined.norm(dim=-1, keepdim=True)

        return combined.cpu().numpy()

    embeddings_data = []
    total_batches = (len(captions_df) + batch_size - 1) // batch_size

    print(f"📊 Processing {len(captions_df)} items in {total_batches} batches")
    print(f"⏱️ Estimated time: ~{total_batches * 2 / 60:.1f} minutes\n")

    for i in tqdm(range(0, len(captions_df), batch_size), desc="Creating embeddings"):
        batch_df = captions_df.iloc[i:i+batch_size]

        # Load images in parallel
        with ThreadPoolExecutor(max_workers=8) as executor:
            images = list(executor.map(load_image_fast, batch_df['image_path']))

        # Filter valid images
        valid_images = []
        valid_indices = []
        for j, img in enumerate(images):
            if img is not None:
                valid_images.append(img)
                valid_indices.append(j)

        if len(valid_images) == 0:
            continue

        # Get captions for valid images
        valid_captions = [batch_df.iloc[idx]['caption'] for idx in valid_indices]

        try:
            # Generate embeddings
            embeddings = generate_embeddings_batch(valid_images, valid_captions)

            # Store results
            for j, idx in enumerate(valid_indices):
                row = batch_df.iloc[idx]
                embeddings_data.append({
                    'image_path': row['image_path'],
                    'filename': row['filename'],
                    'caption': row['caption'],
                    'embedding': embeddings[j].tolist()
                })

        except Exception as e:
            tqdm.write(f"\n⚠️ Batch error: {e}")
            continue

        # Save checkpoint every 1000 items
        if len(embeddings_data) % 1000 == 0 and len(embeddings_data) > 0:
            with open(embeddings_file + '.tmp', 'w') as f:
                json.dump(embeddings_data, f)
            tqdm.write(f"💾 Checkpoint: {len(embeddings_data)} embeddings")

        # Clear cache periodically
        if i % 5000 == 0 and i > 0:
            gc.collect()
            torch.cuda.empty_cache()

    # Save final embeddings
    print(f"\n💾 Saving embeddings...")
    with open(embeddings_file, 'w') as f:
        json.dump(embeddings_data, f)

    # Cleanup
    if os.path.exists(embeddings_file + '.tmp'):
        os.remove(embeddings_file + '.tmp')

    del clip_model
    torch.cuda.empty_cache()
    gc.collect()

    print(f"\n✅ Embedding generation complete!")
    print(f"   Total embeddings: {len(embeddings_data)}")
    print(f"   Success rate: {len(embeddings_data)/len(captions_df)*100:.1f}%")
    print(f"   File: {embeddings_file}")
    print(f"   Size: {os.path.getsize(embeddings_file)/(1024*1024):.2f} MB")

# Verification
print("\n" + "="*60)
print("VERIFICATION")
print("="*60)
if embeddings_data:
    sample = embeddings_data[0]
    print(f"✅ Embedding dimension: {len(sample['embedding'])}")
    print(f"✅ Total items: {len(embeddings_data)}")
    print(f"📋 Sample: {sample['filename']}")
    print(f"   Caption: {sample['caption'][:60]}...")
print("="*60)

STEP 4: ULTRA-FAST CLIP EMBEDDINGS
✅ Found 10097 captions in memory

🚀 Generating CLIP embeddings...
📥 Loading CLIP model on cuda...


100%|████████████████████████████████████████| 338M/338M [00:01<00:00, 290MiB/s]


✅ CLIP loaded

⚙️ Batch size: 128
📊 Processing 10097 items in 79 batches
⏱️ Estimated time: ~2.6 minutes



Creating embeddings: 100%|██████████| 79/79 [01:50<00:00,  1.39s/it]



💾 Saving embeddings...

✅ Embedding generation complete!
   Total embeddings: 10097
   Success rate: 100.0%
   File: /content/fashion_embeddings.json
   Size: 100.03 MB

VERIFICATION
✅ Embedding dimension: 512
✅ Total items: 10097
📋 Sample: img_00000002.jpg
   Caption: Fashion item: a woman in leather pants and a white blouse...


###Store Embeddings in ChromaDB

In [ ]:
import chromadb
from chromadb.config import Settings
import json
import os
from tqdm import tqdm
import uuid

print("="*60)
print("STEP 5: CHROMADB STORAGE")
print("="*60)

# Load embeddings
embeddings_file = '/content/fashion_embeddings.json'

if not os.path.exists(embeddings_file):
    raise Exception("❌ Embeddings not found! Run Step 4 first.")

print("📂 Loading embeddings...")
with open(embeddings_file, 'r') as f:
    embeddings_data = json.load(f)
print(f"✅ Loaded {len(embeddings_data)} embeddings\n")

# Initialize ChromaDB
chroma_path = "/content/chroma_db"
print(f"🗄️ Initializing ChromaDB at {chroma_path}...")

chroma_client = chromadb.PersistentClient(path=chroma_path)

# Delete existing collection if exists
try:
    chroma_client.delete_collection(name="fashion_genome")
    print("🗑️ Deleted existing collection")
except:
    pass

# Create collection
collection = chroma_client.create_collection(
    name="fashion_genome",
    metadata={"description": "Fashion items with CLIP embeddings"}
)
print("✅ Created collection\n")

# Add data in batches
batch_size = 1000
total_batches = (len(embeddings_data) + batch_size - 1) // batch_size

print(f"📊 Adding {len(embeddings_data)} items in {total_batches} batches...")

for i in tqdm(range(0, len(embeddings_data), batch_size), desc="Adding to ChromaDB"):
    batch = embeddings_data[i:i+batch_size]

    documents = []
    embeddings = []
    metadatas = []
    ids = []

    for item in batch:
        documents.append(item['caption'])
        embeddings.append(item['embedding'])
        metadatas.append({
            'image_path': item['image_path'],
            'filename': item['filename']
        })
        ids.append(str(uuid.uuid4()))

    collection.add(
        documents=documents,
        embeddings=embeddings,
        metadatas=metadatas,
        ids=ids
    )

print(f"\n✅ ChromaDB setup complete!")
print(f"   Total items: {collection.count()}")
print(f"   Location: {chroma_path}")
print(f"   Collection: fashion_genome")
print("="*60)

STEP 5: CHROMADB STORAGE
📂 Loading embeddings...
✅ Loaded 10097 embeddings

🗄️ Initializing ChromaDB at /content/chroma_db...
✅ Created collection

📊 Adding 10097 items in 11 batches...


Adding to ChromaDB: 100%|██████████| 11/11 [00:08<00:00,  1.25it/s]



✅ ChromaDB setup complete!
   Total items: 10097
   Location: /content/chroma_db
   Collection: fashion_genome


###Build RAG Retriever with Reranker

In [ ]:
import torch
import clip
from sentence_transformers import CrossEncoder
import numpy as np
from PIL import Image

print("="*60)
print("STEP 6: RAG RETRIEVER + RERANKER")
print("="*60)

# Load CLIP for queries
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"📥 Loading CLIP on {device}...")
clip_model, preprocess = clip.load("ViT-B/32", device=device)
clip_model.eval()
print("✅ CLIP loaded")

# Load reranker
print("📥 Loading reranker...")
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print("✅ Reranker loaded\n")

def retrieve_similar_items(query_text, top_k=20, rerank=True, rerank_top_k=10):
    """
    Retrieve similar fashion items

    Args:
        query_text: Search query
        top_k: Initial results to retrieve
        rerank: Apply reranking
        rerank_top_k: Final number of results after reranking

    Returns:
        Dictionary with results
    """

    # Generate query embedding
    text_tokens = clip.tokenize([query_text]).to(device)
    with torch.no_grad():
        query_embedding = clip_model.encode_text(text_tokens)
        query_embedding = query_embedding / query_embedding.norm(dim=-1, keepdim=True)

    query_list = query_embedding.cpu().numpy().flatten().tolist()

    # Query ChromaDB
    initial_results = collection.query(
        query_embeddings=[query_list],
        n_results=top_k * 2 if rerank else top_k
    )

    if not initial_results['documents'][0]:
        return {
            'query': query_text,
            'documents': [],
            'metadatas': [],
            'distances': [],
            'reranked': False
        }

    # Reranking
    if rerank and len(initial_results['documents'][0]) > 0:
        pairs = [[query_text, doc] for doc in initial_results['documents'][0]]
        rerank_scores = reranker.predict(pairs)

        sorted_indices = np.argsort(rerank_scores)[::-1][:rerank_top_k]

        return {
            'query': query_text,
            'documents': [initial_results['documents'][0][i] for i in sorted_indices],
            'metadatas': [initial_results['metadatas'][0][i] for i in sorted_indices],
            'distances': [initial_results['distances'][0][i] for i in sorted_indices],
            'rerank_scores': [float(rerank_scores[i]) for i in sorted_indices],
            'reranked': True
        }

    return {
        'query': query_text,
        'documents': initial_results['documents'][0][:top_k],
        'metadatas': initial_results['metadatas'][0][:top_k],
        'distances': initial_results['distances'][0][:top_k],
        'reranked': False
    }


def retrieve_by_image(image_path, top_k=20):
    """Retrieve by image"""
    image = Image.open(image_path).convert('RGB')
    image_input = preprocess(image).unsqueeze(0).to(device)

    with torch.no_grad():
        image_embedding = clip_model.encode_image(image_input)
        image_embedding = image_embedding / image_embedding.norm(dim=-1, keepdim=True)

    query_list = image_embedding.cpu().numpy().flatten().tolist()

    results = collection.query(
        query_embeddings=[query_list],
        n_results=top_k
    )

    return {
        'query': f"Image: {image_path}",
        'documents': results['documents'][0],
        'metadatas': results['metadatas'][0],
        'distances': results['distances'][0],
        'reranked': False
    }


# Test retriever
print("🧪 Testing retriever...")
test_queries = [
    "summer casual dresses",
    "formal black suits",
    "vintage denim"
]

for query in test_queries:
    print(f"\n🔍 Query: '{query}'")
    results = retrieve_similar_items(query, top_k=5, rerank=True)
    print(f"   Found: {len(results['documents'])} results")
    if results['documents']:
        print(f"   Top: {results['documents'][0][:70]}...")

print("\n✅ Retriever ready!")
print("="*60)

STEP 6: RAG RETRIEVER + RERANKER
📥 Loading CLIP on cuda...
✅ CLIP loaded
📥 Loading reranker...


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

✅ Reranker loaded

🧪 Testing retriever...

🔍 Query: 'summer casual dresses'
   Found: 10 results
   Top: Fashion item: the best summer dresses for women...

🔍 Query: 'formal black suits'
   Found: 10 results
   Top: Fashion item: a man wearing a black suit and tie...

🔍 Query: 'vintage denim'
   Found: 10 results
   Top: Fashion item: a pair of blue denim shorts...

✅ Retriever ready!


###Trend Analysis with LLM

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

print("="*60)
print("STEP 7: TREND ANALYSIS LLM")
print("="*60)

# Load LLM
llm_name = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"📥 Loading LLM: {llm_name}...")

tokenizer = AutoTokenizer.from_pretrained(llm_name)
llm_model = AutoModelForCausalLM.from_pretrained(
    llm_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)
print("✅ LLM loaded\n")

def generate_quick_summary(retrieved_items, query):
    """Generate quick summary"""
    if not retrieved_items['documents']:
        return f"No items found for: {query}"

    num_items = len(retrieved_items['documents'])
    samples = retrieved_items['documents'][:3]

    summary = f"""**Query:** {query}
**Items Found:** {num_items}
**Reranked:** {'Yes' if retrieved_items['reranked'] else 'No'}

**Top Matches:**
1. {samples[0] if len(samples) > 0 else 'N/A'}
2. {samples[1] if len(samples) > 1 else 'N/A'}
3. {samples[2] if len(samples) > 2 else 'N/A'}
"""
    return summary


def analyze_fashion_trends(retrieved_items, query, max_length=600):
    """Generate comprehensive trend analysis"""

    if not retrieved_items['documents']:
        return f"No items found for query: {query}"

    # Create context
    context_items = []
    for i, doc in enumerate(retrieved_items['documents'][:15]):
        context_items.append(f"{i+1}. {doc}")

    context = "\n".join(context_items)

    prompt = f"""You are a fashion trend analyst. Analyze these fashion items and provide insights.

**Query:** {query}

**Items:**
{context}

**Provide:**
1. **Dominant Trends**: Main patterns, colors, and styles
2. **Color Palette**: Primary colors and combinations
3. **Style Categories**: Types represented
4. **Seasonal Relevance**: Best seasons
5. **Target Demographics**: Who these appeal to
6. **Key Insights**: Notable observations
7. **Styling Tips**: How to wear/combine

Be concise and professional."""

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(llm_model.device)

    with torch.no_grad():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=max_length,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    report = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract generated part
    if "Styling Tips" in report:
        parts = report.split("**Provide:**")
        if len(parts) > 1:
            return parts[1].strip()

    return report


# Test
print("🧪 Testing trend analysis...")
test_query = "summer fashion"
print(f"Query: '{test_query}'")

retrieved = retrieve_similar_items(test_query, top_k=10, rerank=True)
summary = generate_quick_summary(retrieved, test_query)
print(f"\n{summary}")

print("\n✅ Trend analysis ready!")
print("="*60)

STEP 7: TREND ANALYSIS LLM
📥 Loading LLM: Qwen/Qwen2.5-1.5B-Instruct...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ LLM loaded

🧪 Testing trend analysis...
Query: 'summer fashion'

**Query:** summer fashion
**Items Found:** 10
**Reranked:** Yes

**Top Matches:**
1. Fashion item: a pair of shorts and a white t shirt
2. Fashion item: holiday gift guide for her
3. Fashion item: a pair of shoes and a shirt on the floor


✅ Trend analysis ready!


###Streamlit Dashboard

######Create the Full Streamlit App

###Launch the Streamlit App

In [ ]:
!pip install streamlit openai pyngrok pillow -q

In [ ]:
!mkdir -p ~/.streamlit/

!echo "\
[server]\n\
headless = true\n\
port = 8501\n\
enableCORS = false\n\
enableXsrfProtection = false\n\
[theme]\n\
base = 'dark'\n\
" > ~/.streamlit/config.toml

In [ ]:
%%writefile fashion_app.py
import streamlit as st
from openai import OpenAI
import os
import random
from PIL import Image
import json

# Configure the page
st.set_page_config(page_title="Fashion Genome AI", page_icon="👗", layout="wide")

# Dataset path
DATASET_PATH = "/content/fashion_images"

# Title
st.title("👗 Fashion Genome AI - Seasonal Fashion Trends")
st.write("🔮 Ask me about clothing trends and get images from our fashion dataset!")

# Function to get all image files from dataset
@st.cache_data
def load_image_paths():
    image_paths = []
    for root, dirs, files in os.walk(DATASET_PATH):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp')):
                image_paths.append(os.path.join(root, file))
    return image_paths

# Function to categorize images based on keywords
def find_matching_images(query, image_paths, max_images=3):
    """Find images that match the query based on folder/file names"""
    query_lower = query.lower()

    # Define season and category keywords
    season_keywords = {
        'winter': ['winter', 'cold', 'jacket', 'coat', 'sweater', 'boots', 'warm'],
        'summer': ['summer', 'hot', 'shorts', 'dress', 'sandals', 'light', 'beach'],
        'fall': ['fall', 'autumn', 'layer', 'cardigan', 'jeans', 'boots'],
        'spring': ['spring', 'light', 'floral', 'dress', 'casual', 'fresh']
    }

    category_keywords = {
        'dress': ['dress', 'gown', 'frock'],
        'top': ['top', 'shirt', 'blouse', 'tshirt', 't-shirt'],
        'bottom': ['pants', 'jeans', 'skirt', 'shorts', 'trousers'],
        'outerwear': ['jacket', 'coat', 'blazer', 'cardigan'],
        'footwear': ['shoes', 'boots', 'sandals', 'sneakers'],
        'accessories': ['bag', 'hat', 'scarf', 'accessories']
    }

    # Score each image based on relevance
    scored_images = []

    for img_path in image_paths:
        score = 0
        path_lower = img_path.lower()

        # Check for season keywords
        for season, keywords in season_keywords.items():
            if season in query_lower:
                for keyword in keywords:
                    if keyword in path_lower:
                        score += 3

        # Check for category keywords
        for category, keywords in category_keywords.items():
            if category in query_lower:
                for keyword in keywords:
                    if keyword in path_lower:
                        score += 2

        # Check for direct query words in path
        query_words = query_lower.split()
        for word in query_words:
            if len(word) > 3 and word in path_lower:
                score += 1

        if score > 0:
            scored_images.append((img_path, score))

    # Sort by score and return top matches
    scored_images.sort(key=lambda x: x[1], reverse=True)

    # If we have matches, return them
    if scored_images:
        return [img[0] for img in scored_images[:max_images]]

    # If no matches, return random images
    return random.sample(image_paths, min(max_images, len(image_paths)))

# Sidebar for API key input
with st.sidebar:
    st.header("⚙️ Configuration")
    api_key = st.text_input(
        "Enter your OpenAI API Key:",
        type="password",
        help="Get your API key from https://platform.openai.com/api-keys"
    )

    if api_key:
        st.success("✅ API Key configured!")
    else:
        st.warning("⚠️ Please add your OpenAI API key to start chatting")

    st.markdown("---")

    # Image settings
    show_images = st.checkbox("🖼️ Show Images from Dataset", value=True)
    num_images = st.slider("Number of images to show", 1, 6, 3)

    st.markdown("---")
    st.markdown("### 📋 Quick Questions")

    # Quick question buttons
    if st.button("❄️ Winter Fashion Trends"):
        st.session_state.pending_prompt = "What are the latest winter fashion trends? Show me winter outfits."
        st.rerun()

    if st.button("☀️ Summer Fashion Trends"):
        st.session_state.pending_prompt = "What are the trending clothes for summer? Show me summer styles."
        st.rerun()

    if st.button("🍂 Fall/Autumn Trends"):
        st.session_state.pending_prompt = "What should I wear in fall? Show me autumn fashion."
        st.rerun()

    if st.button("🌸 Spring Fashion Trends"):
        st.session_state.pending_prompt = "What are the spring fashion trends? Show me spring outfits."
        st.rerun()

    st.markdown("---")

    # Load and display dataset info
    all_images = load_image_paths()
    st.info(f"📂 Dataset: {len(all_images)} images loaded")

    st.markdown("---")
    st.markdown("### 💡 Example Questions:")
    st.markdown("""
    - Show me winter jackets and coats
    - What are summer dress trends?
    - Display fall casual outfits
    - Show spring fashion accessories
    - Trending winter boots styles
    - Summer casual wear ideas
    """)

    st.markdown("---")
    if st.button("🗑️ Clear Chat History"):
        st.session_state.messages = []
        st.session_state.user_input = ""
        st.rerun()

# Initialize session state
if "messages" not in st.session_state:
    st.session_state.messages = []

if "user_input" not in st.session_state:
    st.session_state.user_input = ""

if "pending_prompt" not in st.session_state:
    st.session_state.pending_prompt = None

# Display chat history
chat_container = st.container()
with chat_container:
    for message in st.session_state.messages:
        with st.chat_message(message["role"]):
            if message["role"] == "assistant":
                # Display text content
                if "content" in message:
                    st.markdown(message["content"])

                # Display images if present
                if "image_paths" in message and message["image_paths"]:
                    st.markdown("---")
                    st.markdown("### 📸 From Our Fashion Collection:")

                    # Display images in grid
                    cols = st.columns(min(3, len(message["image_paths"])))
                    for idx, img_path in enumerate(message["image_paths"]):
                        try:
                            with cols[idx % 3]:
                                img = Image.open(img_path)
                                st.image(img, use_container_width=True)
                                # Show filename as caption
                                filename = os.path.basename(img_path)
                                st.caption(filename)
                        except Exception as e:
                            st.error(f"Error loading image: {e}")
            else:
                st.markdown(message["content"])

# Show helpful info if no messages yet
if len(st.session_state.messages) == 0:
    st.info("👋 Welcome! Ask about fashion trends and see real images from our dataset!")

    # Display some example cards
    col1, col2, col3 = st.columns(3)

    with col1:
        st.markdown("### 🎨 Seasonal Trends")
        st.markdown("Explore winter, summer, fall, spring styles")

    with col2:
        st.markdown("### 👔 Outfit Categories")
        st.markdown("Browse dresses, tops, bottoms, outerwear")

    with col3:
        st.markdown("### 🧥 Real Fashion")
        st.markdown("Images from our curated collection")

# Create input section at the bottom
st.markdown("---")
input_col1, input_col2 = st.columns([6, 1])

with input_col1:
    user_input = st.text_input(
        "Your question:",
        value=st.session_state.user_input,
        placeholder="Ask about fashion trends... (e.g., 'Show me winter jackets')",
        key="input_field",
        label_visibility="collapsed"
    )

with input_col2:
    send_button = st.button("Send 🚀", use_container_width=True)

# Handle pending prompt from quick buttons
prompt = None
if st.session_state.pending_prompt:
    prompt = st.session_state.pending_prompt
    st.session_state.pending_prompt = None
elif send_button and user_input.strip():
    prompt = user_input
    st.session_state.user_input = ""

# Process the prompt
if prompt:
    # Check if API key is provided
    if not api_key:
        st.error("⚠️ Please enter your OpenAI API Key in the sidebar to start chatting!")
        st.info("👉 Get your API key from: https://platform.openai.com/api-keys")
        st.stop()

    # Add user message to chat history
    st.session_state.messages.append({"role": "user", "content": prompt})

    # Display user message
    with chat_container:
        with st.chat_message("user"):
            st.markdown(prompt)

    # Generate response
    with chat_container:
        with st.chat_message("assistant"):
            message_placeholder = st.empty()
            image_container = st.container()

            try:
                # Configure OpenAI API client
                client = OpenAI(api_key=api_key)

                # Create fashion-focused system message
                system_message = """You are an expert fashion consultant and stylist with deep knowledge of seasonal trends,
                clothing styles, color theory, and fashion history. You provide personalized, practical fashion advice.

                Provide comprehensive, friendly, and actionable responses that include:
                1. Current Trends: Latest fashion trends relevant to the query
                2. Specific Items: Concrete clothing pieces and styles
                3. Color Palette: Trending colors and combinations
                4. Fabrics & Materials: Recommended textiles for the season/occasion
                5. Styling Tips: How to put together outfits and accessorize

                Be specific, enthusiastic, and helpful. Keep responses concise but informative."""

                # Prepare messages for API
                messages = [{"role": "system", "content": system_message}]

                # Add conversation history
                for msg in st.session_state.messages[-6:]:
                    if msg["role"] in ["user", "assistant"]:
                        messages.append({"role": msg["role"], "content": msg.get("content", "")})

                # Generate text response
                with st.spinner("✨ Analyzing fashion trends..."):
                    stream = client.chat.completions.create(
                        model="gpt-4o-mini",
                        messages=messages,
                        temperature=0.7,
                        max_tokens=1000,
                        stream=True
                    )

                    full_response = ""
                    for chunk in stream:
                        if chunk.choices[0].delta.content is not None:
                            full_response += chunk.choices[0].delta.content
                            message_placeholder.markdown(full_response + "▌")

                    message_placeholder.markdown(full_response)

                # Find and display matching images from dataset
                selected_images = []
                if show_images:
                    with st.spinner("🔍 Finding matching images from dataset..."):
                        all_images = load_image_paths()
                        selected_images = find_matching_images(prompt, all_images, num_images)

                        if selected_images:
                            with image_container:
                                st.markdown("---")
                                st.markdown("### 📸 From Our Fashion Collection:")

                                # Display images in grid
                                cols = st.columns(min(3, len(selected_images)))
                                for idx, img_path in enumerate(selected_images):
                                    try:
                                        with cols[idx % 3]:
                                            img = Image.open(img_path)
                                            st.image(img, use_container_width=True)
                                            filename = os.path.basename(img_path)
                                            st.caption(filename)
                                    except Exception as e:
                                        st.error(f"Error loading image: {e}")

                # Add assistant response to chat history
                st.session_state.messages.append({
                    "role": "assistant",
                    "content": full_response,
                    "image_paths": selected_images
                })

            except Exception as e:
                error_message = f"""❌ **Error occurred**: {str(e)}

**Troubleshooting:**
1. Check if your OpenAI API key is correct
2. Make sure dataset is properly extracted
3. Ensure you have API credits available

Please fix the issue and try again."""
                message_placeholder.error(error_message)
                st.session_state.messages.append({"role": "assistant", "content": error_message})

    # Rerun to update UI
    st.rerun()

# Footer
st.markdown("---")
st.markdown("💡 **Pro Tip:** Mention seasons (winter, summer, fall, spring) or categories (dress, jacket, shoes) for better image matches!")
st.markdown(f"📂 **Dataset:** {len(load_image_paths())} fashion images available")

Writing fashion_app.py


In [ ]:
from pyngrok import ngrok
import time

# Kill any existing ngrok tunnels
ngrok.kill()

# REPLACE WITH YOUR NGROK AUTHTOKEN
NGROK_AUTH_TOKEN = ""  

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Create tunnel
public_url = ngrok.connect(8501)
print("🔥 Public URL:", public_url)
print("\n✅ Click the link above to access your Fashion Chatbot")
print("⏳ Wait 5-10 seconds for Streamlit to fully load...")

# Run Streamlit
!streamlit run fashion_app.py --server.port 8501 --server.headless true

🔥 Public URL: NgrokTunnel: "https://prosodemic-pia-skirtless.ngrok-free.dev" -> "http://localhost:8501"

✅ Click the link above to access your Fashion Chatbot
⏳ Wait 5-10 seconds for Streamlit to fully load...
Error parsing config toml. This is most likely due to a syntax error in the config.toml file. Please fix it and try again.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/streamlit/config.py", line 2449, in _update_config_with_toml
    parsed_config_file = toml.loads(raw_toml)
                         ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/toml/decoder.py", line 433, in loads
    raise TomlDecodeError("Key group not on a line by itself.",
toml.decoder.TomlDecodeError: Key group not on a line by itself. (line 1 column 1 char 0)



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.247.156.238:8501

